In [ ]:
# Cell 1 -- Setup + download NewsEdits AP data
# Requires: Wikipedia training already done (wiki_model.pt must exist)
# Run kaggle_notebook.ipynb first if checkpoint is missing.

!pip install transformers scikit-learn datasets scipy gdown -q

import os, sys, glob
from pathlib import Path

# Clone / pull repo
REPO = Path('/kaggle/working/multihop-retrieval')
if REPO.exists():
    os.system(f'git -C {REPO} pull')
else:
    os.system(f'git clone https://github.com/haaaaaaayrewugrhyw/multihop-retrieval.git {REPO}')
sys.path.insert(0, str(REPO / 'delta_system'))
print('Repo OK. Files:', sorted([f.name for f in (REPO/'delta_system').glob('*.py')]))

# Check for existing checkpoint
ckpts = glob.glob('/kaggle/working/checkpoints/*.pt')
print('Checkpoints found:', ckpts)

# Download NewsEdits AP data via gdown
# Google Drive: https://drive.google.com/drive/folders/17a5S3liA0C91XbgnMBUQBo-NVb22Z9xf
# Subfolder: matched-sentences/ -> ap-matched-sentences.db

DB_PATH = Path('/kaggle/working/newsedits/ap-matched-sentences.db')

if not DB_PATH.exists():
    print()
    print('Downloading NewsEdits (AP matched-sentences folder)...')
    print('This downloads from Google Drive -- may take a few minutes.')
    DB_PATH.parent.mkdir(parents=True, exist_ok=True)
    
    # Download the matched-sentences subfolder from NewsEdits Drive
    # Folder ID: 17a5S3liA0C91XbgnMBUQBo-NVb22Z9xf
    import gdown
    try:
        gdown.download_folder(
            id='17a5S3liA0C91XbgnMBUQBo-NVb22Z9xf',
            output='/kaggle/working/newsedits_raw/',
            quiet=False,
            remaining_ok=True
        )
        # Find any .db file downloaded
        db_files = glob.glob('/kaggle/working/newsedits_raw/**/*.db', recursive=True)
        print(f'Downloaded .db files: {db_files}')
        if db_files:
            # Use the first one found (prefer AP)
            ap_files = [f for f in db_files if 'ap' in f.lower()]
            chosen = ap_files[0] if ap_files else db_files[0]
            os.system(f'cp "{chosen}" "{DB_PATH}"')
            print(f'Using: {chosen}')
        else:
            print('No .db files found. See manual instructions below.')
    except Exception as e:
        print(f'gdown failed: {e}')
        print()
        print('MANUAL DOWNLOAD INSTRUCTIONS:')
        print('1. Go to: https://drive.google.com/drive/folders/17a5S3liA0C91XbgnMBUQBo-NVb22Z9xf')
        print('2. Open the matched-sentences/ subfolder')
        print('3. Download ap-matched-sentences.db (or any .db file)')
        print('4. In Kaggle: click + Add Data -> Upload -> upload the .db file')
        print('5. Update DB_PATH below to point to the uploaded file')
        print('6. Re-run this cell and Cell 2')
else:
    print(f'Database already exists: {DB_PATH}')

print()
print(f'DB_PATH exists: {DB_PATH.exists()}')
if DB_PATH.exists():
    size_mb = DB_PATH.stat().st_size / 1e6
    print(f'File size: {size_mb:.1f} MB')

In [ ]:
# Cell 2 -- Run zero-shot NewsEdits evaluation
# Loads Wikipedia checkpoint, evaluates on news revision pairs, prints AUC vs TF-IDF

# If you manually uploaded the .db file, update this path:
DB_PATH = '/kaggle/working/newsedits/ap-matched-sentences.db'

!python /kaggle/working/multihop-retrieval/delta_system/newsedits_zeroshot_eval.py \
    --db {DB_PATH} \
    --ckpt /kaggle/working/checkpoints/wiki_model.pt \
    --n 1000 \
    --min_added 2